# CMT — ARCADE Inpainting Pipeline

Train the CMT inpainting model on vessel-masked coronary angiography X-rays.

---
⚠️ **Before running:** Runtime → Change runtime type → **T4 GPU**

**New features:**
- Enhanced metrics: PSNR, SSIM, Wasserstein Distance, RMSE
- Improved visualization with adaptive mask detection
- Automated Google Drive checkpoint mirroring
- Makefile-based workflow

In [ ]:
# ── 1. Check GPU ──────────────────────────────────────────────────────────
!nvidia-smi

In [ ]:
# ── 2. Clone repo ─────────────────────────────────────────────────────────
!git clone https://github.com/C0d3Crush/arcade-xray-inpainting.git
%cd /content/arcade-xray-inpainting

In [ ]:
# ── 3. Install dependencies ───────────────────────────────────────────────
!pip install -q -r requirements.txt

In [ ]:
# ── 4. Mount Google Drive & unzip ARCADE dataset ─────────────────────────
from google.colab import drive
drive.mount('/content/drive')

# Check if arcade.zip exists and find correct path
import os
possible_paths = [
    '/content/drive/MyDrive/CMT/arcade.zip',           # English Drive
    '/content/drive/Meine Ablage/CMT/arcade.zip',     # German Drive  
    '/content/drive/MyDrive/Meine Ablage/CMT/arcade.zip',
    '/content/drive/MyDrive/arcade.zip'
]

ARCADE_ZIP = None
for path in possible_paths:
    if os.path.exists(path):
        ARCADE_ZIP = path
        print(f"✓ Found ARCADE dataset: {path}")
        break

if ARCADE_ZIP is None:
    print("❌ ARCADE dataset not found. Please upload arcade.zip to Google Drive.")
    print("Expected locations:")
    for path in possible_paths:
        print(f"  - {path}")
    # List actual Drive contents for debugging
    print("\nActual Drive contents:")
    !ls -la "/content/drive/"
    !ls -la "/content/drive/MyDrive/" 2>/dev/null || echo "MyDrive not found"
else:
    # Unzip - check internal structure first
    !unzip -l "$ARCADE_ZIP" | head -20
    
    # Unzip to current directory, then move to data/
    !unzip -q -o "$ARCADE_ZIP"
    !mkdir -p data
    !mv arcade data/ 2>/dev/null || echo "Already in correct location"
    
    # Check final structure
    !find data/ -name '*.json' | head -5
    !ls -la data/arcade/syntax/ 2>/dev/null || ls -la data/

---
## Setup & Preprocessing

### Cache Masks (Optional - Speeds up training)

In [ ]:
# ── 5. Cache masks and annotations for faster training ────────────────────
!make cache-data

# This will:
# 1. Precompute vessel masks from COCO annotations (train + val)
# 2. Create .pkl files for 10x faster annotation loading
# 3. Store in data/masks_cache/ for optional use with --train_mask flag

---
## Train CMT Inpainting Model

### Setup Google Drive Mirroring (Optional)

In [ ]:
# ── 6a. Setup Google Drive checkpoint mirroring ──────────────────────────
!mkdir -p /content/drive/MyDrive/CMT/checkpoints

# ── 6b. Train CMT (with enhanced metrics) ──────────────────────────────────
!python src/train.py \
    --train_img  data/arcade/syntax/train/images \
    --train_ann  data/arcade/syntax/train/annotations/train.json \
    --val_img    data/arcade/syntax/val/images \
    --val_ann    data/arcade/syntax/val/annotations/val.json \
    --input_size 256 \
    --epochs     100 \
    --batch_size 16 \
    --device     cuda \
    --drive_ckpt /content/drive/MyDrive/CMT/checkpoints

# Alternative: Use Makefile (requires smaller input size for Colab)
# !make train ARGS="--input_size 128 --batch_size 8 --epochs 50 --device cuda"

---
## Results & Visualization

### Plot Training Metrics

In [ ]:
# ── 7a. Plot training metrics ──────────────────────────────────────────────
!python scripts/plot_training.py outputs/checkpoints/training_log.csv

# Display the plot
from IPython.display import Image, display
display(Image('training_plot.png'))

# ── 7b. Generate test samples and visualizations ───────────────────────────
!make prepare-samples
!make inference
!make visualize

# Display comparison results
import os
comparison_dir = 'outputs/samples/comparisons'
if os.path.exists(comparison_dir):
    for img_file in sorted(os.listdir(comparison_dir))[:3]:  # Show first 3
        if img_file.endswith('.png'):
            print(f"\n--- {img_file} ---")
            display(Image(os.path.join(comparison_dir, img_file)))

---
## Download Checkpoint

### Download from Colab

In [ ]:
# ── 8. Download best checkpoint ───────────────────────────────────────────
from google.colab import files

# Download checkpoint (automatically mirrored to Google Drive)
files.download('outputs/checkpoints/best.pth')
files.download('outputs/checkpoints/training_log.csv')

print("\n✓ Checkpoint also available in Google Drive: /MyDrive/CMT/checkpoints/")

---
## Local Setup Instructions

### After Download

```bash
# Move downloaded files
mv ~/Downloads/best.pth outputs/checkpoints/
mv ~/Downloads/training_log.csv outputs/checkpoints/

# Local inference
python src/demo.py \
  --ckpt outputs/checkpoints/best.pth \
  --img_path outputs/samples/test_img \
  --mask_path outputs/samples/test_mask \
  --output_path outputs/samples/results \
  --device cpu

# Visualize results
make visualize

# Plot training metrics
make plot
```

### Training Metrics

The model now tracks enhanced metrics:
- **PSNR** (Peak Signal-to-Noise Ratio) - Higher is better
- **SSIM** (Structural Similarity Index) - Higher is better  
- **Wasserstein Distance** - Lower is better (Earth Mover's Distance)
- **RMSE** (Root Mean Square Error) - Lower is better